# Proyecto RL DEFI IV

**Integrantes**: Juan Camilo y Jesús Figueroa

**Descripción**:

Creación de un agente de aprendizaje por refuerzo para operar el token POL, empleando el histórico de transacciones on-chain y métricas financieras para identificar wallets con desempeño consistente, construir un perfil estadístico de su comportamiento e incorporar dicho perfil al proceso de toma de decisiones del agente, con el propósito de mejorar las decisiones de compra, venta o hold del activo.



El sistema reconstruirá el historial de actividad de las wallets seleccionadas, se usaran el precio de cierre de POL-USD en ventanas de 1m, 5m, 15m y 1h, con el objetivo de calcular la ganancia generada en esas ventanas.

## Guía de ejecución reproducible (paso a paso)

Esta guía permite ejecutar el proyecto desde Jupyter sin repetir descargas innecesarias. La ruta recomendada para la sustentación es reutilizar los archivos Parquet de `cache/`; sólo usa Alchemy cuando realmente necesites una ventana nueva de swaps.

### 1. Preparar el entorno

En una terminal abierta en la carpeta del repositorio ejecuta:

```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter lab
```

Abre este archivo (`Defi_IV.ipynb`) y selecciona el kernel de `.venv`. Después de instalar dependencias por primera vez, reinicia el kernel.

### 2. Elegir la fuente de datos

**Opción recomendada — sin red:** comprueba que existan `cache/swaps_24h.parquet` y, si vas a mostrar el resumen por wallet, `cache/wallets_24h.parquet`. Con `forzar_descarga=False`, el `SwapPipeline` reutiliza el cache y no consulta Alchemy.

**Opción para datos nuevos:** antes de ejecutar la configuración, define `ALCHEMY_RPC_URL` o `ALCHEMY_API_KEY` como variable de entorno. Si no hay Parquet, la celda de configuración solicitará la clave de forma oculta. Nunca pegues ni guardes la clave en el notebook ni en GitHub.

### 3. Ejecutar los perfiles de wallets

Ejecuta, en este orden: **Imports** → **Main Transacciones** (configuración y swaps) → **Main Wallets** → la celda que crea `resultado`. Esta última genera el informe y expone `resultado.perfiles`, `resultado.ganadoras` y `resultado.snapshot_dir`.

Luego ejecuta las celdas Buy, Sell y Hold para explorar los perfiles ganadores por horizonte. Una wallet entra como ganadora sólo si tiene decisiones maduras suficientes, PnL y retorno mediano positivos y el `consistency_score` requerido.

### 4. Ejecutar el agente Bellman

Desde **Parte III**, ejecuta las celdas en orden. La celda `ejecutar_rl_desde_snapshot` recibe `resultado.snapshot_dir`: reutiliza los artefactos ya generados y no consulta Alchemy. Finalmente, las últimas celdas muestran estados, validación de transiciones, política y el replay de riqueza.

### 5. Comprobar los artefactos

- `informes_pol/...`: perfiles, ganadoras e informe HTML.
- `informes_rl/...`: observaciones, política Bellman, matriz de transiciones y replay histórico.
- Para repetir la ejecución con los mismos datos, conserva `cache/` y vuelve a ejecutar desde la sección **Imports**.


## Imports

In [1]:
from defi4.data import SwapPipeline
from defi4.data import WalletView
import warnings
warnings.filterwarnings('ignore')

## Main Transacciones

In [2]:
import os
from getpass import getpass
from pathlib import Path

POOL_ADDRESS = "0xA374094527e1673A86dE625aa59517c5dE346d32"  # WPOL/USDC 0.05%
TICKER_YF    = "POL28321-USD"
DECIMALES_T0 = 18   # WPOL
DECIMALES_T1 = 6    # USDC
NOMBRE_T0    = "pol"
NOMBRE_T1    = "usdc"

HORAS        = 24
HORIZONTES   = ["1m", "5m", "15m", "1h"]
CACHE_PATH   = "cache/swaps_24h.parquet"

# Con cache existente no se requiere conexión a Alchemy.
RPC_URL = os.environ.get("ALCHEMY_RPC_URL", "").strip()
if not RPC_URL:
    api_key = os.environ.get("ALCHEMY_API_KEY", "").strip()
    if not api_key and not Path(CACHE_PATH).is_file():
        api_key = getpass("Clave de Alchemy (no se guarda): ")
    if api_key:
        RPC_URL = f"https://polygon-mainnet.g.alchemy.com/v2/{api_key}"

if not Path(CACHE_PATH).is_file() and not RPC_URL:
    raise RuntimeError("No hay cache y falta ALCHEMY_RPC_URL o ALCHEMY_API_KEY.")


In [3]:
pipeline = SwapPipeline(
    rpc_url      = RPC_URL,
    pool_address = POOL_ADDRESS,
    ticker_yf    = TICKER_YF,
    decimales_t0 = DECIMALES_T0,
    decimales_t1 = DECIMALES_T1,
    nombre_t0    = NOMBRE_T0,
    nombre_t1    = NOMBRE_T1,
    cache_path   = CACHE_PATH,
)

swaps = pipeline.ejecutar(
    horas            = HORAS,
    horizontes       = HORIZONTES,
    forzar_descarga  = False,
    verbose          = True,
)


[cache] cargado ← cache\swaps_24h.parquet (21331 swaps)
[yfinance] descargando velas 1m…
[yfinance] descargando velas 5m…
[yfinance] descargando velas 15m…
[yfinance] descargando velas 1h…


In [7]:
# Tabla de Swaps
swaps.head()

,timestamp,bloque,hash_tx,wallet,direccion,pol_cantidad,usdc_cantidad,precio_ejecutado,gas_pol,gas_usdc,...,ganancia_usdc_1m,precio_fwd_5m,retorno_5m,ganancia_usdc_5m,precio_fwd_15m,retorno_15m,ganancia_usdc_15m,precio_fwd_1h,retorno_1h,ganancia_usdc_1h
0,2026-08-21 20:47:52-05:00,92438830,0xba45eff7940a3e13a3efe6650f54d9f9fd7995d84a24...,0x49fB304bA92f732486d23DE10d5824ff67A85F75,Sell,56.976172,5.453435,0.095714,0.240044,0.022976,...,-0.004882,0.09531,0.004224,0.023036,0.09616,-0.004656,-0.025394,0.09907,-0.035059,-0.191194
1,2026-08-21 20:48:36-05:00,92438859,0x8cc4fd5c05fd576cfd6d216e2231d7b89cfa8f915de9...,0xe27D4Cb1759BAE0188126bD62D415a6Fa528FFE5,Buy,210.691960,20.189782,0.095826,1.328537,0.127308,...,-0.051845,0.09531,-0.005385,-0.108731,0.09616,0.003485,0.070357,0.09907,0.033852,0.683470
2,2026-08-21 20:49:03-05:00,92438877,0x42520a2fd84fbe4ed8b9cc301cbb319a0497410ff00e...,0xeba3A54cffcC11123578C5A740a45456d5C6D4a8,Sell,222.132001,21.264494,0.095729,0.071096,0.006806,...,0.033118,0.09531,0.004378,0.093092,0.09616,-0.004501,-0.095720,0.09907,-0.034900,-0.742123
3,2026-08-21 20:49:12-05:00,92438883,0x02a30b4fb5bcc347379ffecbe0211c4ec566e2d9db77...,0xeba3A54cffcC11123578C5A740a45456d5C6D4a8,Sell,191.333283,18.307962,0.095686,0.070256,0.006722,...,0.020327,0.09531,0.003932,0.071986,0.09616,-0.004951,-0.090647,0.09907,-0.035363,-0.647426
4,2026-08-21 20:49:52-05:00,92438910,0x0087aed6f5d9d6a2d612db08ac0246dd39bb1eeff959...,0xeba3A54cffcC11123578C5A740a45456d5C6D4a8,Sell,224.353321,21.457538,0.095642,0.086444,0.008268,...,0.013848,0.09531,0.003468,0.074422,0.09616,-0.005419,-0.116278,0.09907,-0.035845,-0.769145


## Main Wallets

In [4]:
# Para Historial de Wallets
wallets = WalletView(pipeline, horizontes=HORIZONTES)

In [5]:
# Historial y tabla agregada que alimenta el informe por wallet
from pathlib import Path

wallets.construir()
Path("cache").mkdir(exist_ok=True)
wallets.df.to_parquet("cache/wallets_24h.parquet", index=False)
print("Tabla de wallets guardada en cache/wallets_24h.parquet")


,wallet,direccion,n_swaps,pol_cantidad,usdc_cantidad,gas_usdc,usdc_neto,ganancia_neta_usdc_1m,ganancia_neta_usdc_5m,ganancia_neta_usdc_15m,ganancia_neta_usdc_1h
0,0x003AB61c1f180Eee803b9EA334B6b1B29A4c8364,Sell,1,777.660000,84.096490,0.029387,84.067103,-0.122368,-0.977797,0.484208,0.095377
1,0x004125dc64f050C3CCEaD24A61c89aDDD4C2c794,Buy,1,111.208914,11.962802,0.047072,11.915730,-0.081605,-0.078270,-0.099399,-0.352956
2,0x0051391747B6C11110dA7dE76B3F4AED532d7E40,Buy,1,121.204415,12.272375,0.062058,12.210317,-0.117028,0.329004,0.579898,1.261067
3,0x007d9668D19418b66CD028cde1Ea598260152F93,Buy,1,13.590998,1.495507,0.042197,1.453310,-0.038889,-0.038753,-0.018910,-0.047587
4,0x0082CaF47363bD42917947d81f4d4E0395257267,Buy,1,137.456061,14.998583,0.181089,14.817494,-0.239573,-0.521357,-0.558471,0.238775
...,...,...,...,...,...,...,...,...,...,...,...
3053,0xfd50D4D48896e076A02Bd5F7534df1244948eAAE,Sell,1,282.750000,30.054951,0.012830,30.042121,0.064965,-0.260196,-0.279988,-0.427018
3054,0xfeA573eF9d5eD5b470311582C99c103C38C0320f,Buy,1,0.002269,0.000243,0.020015,0.000000,-0.020016,-0.020015,-0.020018,-0.020022
3055,0xff7D510CF1ec1d376DB53e45bC754bDd8c041D3e,Buy,1,8.954345,1.032616,0.113928,0.918688,-0.109094,-0.095483,-0.081783,-0.133181
3056,0xff91d71CdbA995Bb2F2Ad009F2309E542036Ebd2,Buy,1,15.822873,1.739953,0.043565,1.696388,-0.053604,-0.039363,-0.036515,-0.027496


In [9]:
from defi4.pipeline import ejecutar_desde_parquet

resultado = ejecutar_desde_parquet(
    parquet_path="cache/swaps_24h.parquet",  # ajusta si tu archivo tiene otro nombre
    tabla_wallets="cache/wallets_24h.parquet",
    output_dir="informes_pol",
    export_png=False,
)

Informe creado sin red: informes_pol\snapshot_20260823T014719Z_06


## WALLETS GANADORAS 

In [11]:
perfiles = resultado.perfiles
perfiles["winner_status"].value_counts()


insufficient    11220
not_winner       1573
winner            631
pending            54
Name: winner_status, dtype: int64

In [10]:
display(
    perfiles[perfiles["winner_status"] == "winner"]
    .sort_values("consistency_score", ascending=False)
)


,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
12847,0x064ad6bf448424ebc78e5f315b9cc8001ddb5f7c,HOLD,1h,17,0,7.817119,0.009126,1.000000,3.000000,1.000000,winner,NET_SELLER_SWING,1.311944
12852,0x890e77985a42d9bb282e195a688005459df8fd51,HOLD,5m,11,0,13.093970,0.011043,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,2.913333
12855,0xe8696676836ec709ac693eab66c4cfc287b64496,HOLD,5m,21,0,28.965624,0.014772,1.000000,3.000000,1.000000,winner,NET_SELLER_SWING,1.980833
12854,0xd0d08887e8a5b16049534a7f3fc1de92848f5bea,HOLD,1h,75,0,86.655102,0.041896,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,8.319444
12853,0xd0d08887e8a5b16049534a7f3fc1de92848f5bea,HOLD,15m,75,0,30.525063,0.013723,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,8.319444
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13473,0xb2bffc3788acfb35bf6555fa1fe6adc2c0afe9e5,BUY_POL,1m,120,0,1.857759,0.000453,0.575000,1.178249,0.605325,winner,NET_SELLER_SCALPER,0.717222
13474,0xf78ca0e0be3697c74121e54fb8163ed7d33c8e9d,BUY_POL,15m,3,0,0.245220,0.022977,0.666667,2.116220,0.604955,winner,NET_BUYER_UNKNOWN_HOLD,NaN
13475,0x3fc0910bebc34db4c985474c4dd2f84170aee276,SELL_POL,1h,3,0,2.210758,0.013549,0.666667,2.099946,0.603328,winner,NET_BUYER_SWING,1.042639
13476,0xb886a6f9725b5c387f221b13d0efda1e66999aef,SELL_POL,5m,19,0,7.370357,0.000492,0.526316,1.401371,0.603295,winner,NET_BUYER_SWING,5.603889


## ACCION COMPRA 

In [17]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_buy_1m= filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["BUY_POL"],
    horizontes=["1m"],
)

display(solo_buy_1m)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0x890e77985a42d9bb282e195a688005459df8fd51,BUY_POL,1m,8,0,8.557185,0.002157,1.000000,3.000000,0.960000,winner,NET_BUYER_SWING,2.913333
1,0x33b77e6a2a88a1bae3e27b7e6bd5a34dc19dd5ee,BUY_POL,1m,3,0,9.255996,0.011744,1.000000,3.000000,0.860000,winner,NET_BUYER_UNKNOWN_HOLD,NaN
2,0xde00000912e17b8b0a13760a8525fed73352dfa7,BUY_POL,1m,3,0,0.603910,0.002171,1.000000,3.000000,0.860000,winner,NET_BUYER_UNKNOWN_HOLD,NaN
3,0x1daa4f9e8f68b71541ed4eb08f496c8e591a6bb5,BUY_POL,1m,3,0,0.521472,0.003492,1.000000,3.000000,0.860000,winner,NET_SELLER_SCALPER,0.513056
4,0xaaa118f5ee455be7e65c37782dde8a31c81f35a3,BUY_POL,1m,3,0,0.471573,0.002158,1.000000,3.000000,0.860000,winner,NET_BUYER_SCALPER,0.811667
5,0xae98f425b4fb3b208160ec31903b3def960e1602,BUY_POL,1m,3,0,0.317246,0.000379,1.000000,3.000000,0.860000,winner,NET_SELLER_SWING,2.975556
6,0xce33333daf1f148e0e944a782105f06856b89c08,BUY_POL,1m,3,0,0.205851,0.004009,1.000000,3.000000,0.860000,winner,NET_SELLER_SCALPER,0.428333
7,0xde0000021207f471c5890e985dfad1012f963be6,BUY_POL,1m,3,0,0.109998,0.000749,1.000000,3.000000,0.860000,winner,NET_BUYER_UNKNOWN_HOLD,NaN
8,0xe8696676836ec709ac693eab66c4cfc287b64496,BUY_POL,1m,12,0,11.306734,0.001912,0.666667,3.000000,0.833333,winner,NET_SELLER_SWING,1.980833
9,0x555c98b0cc6f301cadeead302d380545ab84e031,BUY_POL,1m,5,0,7.651706,0.004111,0.800000,3.000000,0.800000,winner,NET_BUYER_SWING,1.532500


In [18]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_buy_5m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["BUY_POL"],
    horizontes=["5m"],
)

display(solo_buy_5m)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0xe8696676836ec709ac693eab66c4cfc287b64496,BUY_POL,5m,12,0,29.803036,0.011272,0.916667,3.000000,0.958333,winner,NET_SELLER_SWING,1.980833
1,0xe17cd5ee4152e195d4ed389ad575d09f3c3e980d,BUY_POL,5m,10,0,4.371264,0.004176,0.800000,3.000000,0.900000,winner,NET_BUYER_SWING,5.460278
2,0x890e77985a42d9bb282e195a688005459df8fd51,BUY_POL,5m,8,0,14.776225,0.009257,0.875000,3.000000,0.897500,winner,NET_BUYER_SWING,2.913333
3,0xce33333ae1d7e5a5f4af479e3b098b6b31d67413,BUY_POL,5m,4,0,6.171436,0.015328,1.000000,3.000000,0.880000,winner,NET_BUYER_UNKNOWN_HOLD,NaN
4,0x133edea1cafc3135fa60ea613ef908a0534859a4,BUY_POL,5m,4,0,1.953969,0.010354,1.000000,3.000000,0.880000,winner,NET_BUYER_UNKNOWN_HOLD,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,0xde000009803dfd691e50ecfd6314950c660e7480,BUY_POL,5m,4,0,2.626036,0.001407,0.500000,3.000000,0.630000,winner,NET_BUYER_SWING,1.932222
71,0xc066ac5d385419b1a8c43a0e146fa439837a8b8c,BUY_POL,5m,4,0,0.958839,0.002310,0.500000,3.000000,0.630000,winner,NET_BUYER_UNKNOWN_HOLD,NaN
72,0xf86cdcadb5e56f6aebece7986c8642cf4b050484,BUY_POL,5m,6,0,0.282297,0.006617,0.666667,1.633905,0.616724,winner,NET_BUYER_SWING,5.093472
73,0x05291cfeb01f027044a88a3ced310ef903aeca10,BUY_POL,5m,23,0,0.999052,0.001440,0.521739,1.554705,0.616340,winner,NET_SELLER_SWING,1.110417


In [19]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_buy_15m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["BUY_POL"],
    horizontes=["15m"],
)

display(solo_buy_15m)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0x09f2f9b23e1006546ef45e5993ebef224cb4d4cd,BUY_POL,15m,13,0,6.004493,0.014713,0.923077,3.000000,0.961538,winner,NET_BUYER_SWING,1.620833
1,0x5f4674311dcd7b3e29e9cfa396d20d21b7135f2b,BUY_POL,15m,6,0,20.566266,0.023075,1.000000,3.000000,0.920000,winner,NET_BUYER_SCALPER,0.949028
2,0xe8696676836ec709ac693eab66c4cfc287b64496,BUY_POL,15m,12,0,46.455135,0.019257,0.833333,3.000000,0.916667,winner,NET_SELLER_SWING,1.980833
3,0x19b5ab374071703813e8f67f2762a93e2134796c,BUY_POL,15m,17,0,13.116682,0.016789,0.823529,3.000000,0.911765,winner,NET_BUYER_UNKNOWN_HOLD,NaN
4,0x890e77985a42d9bb282e195a688005459df8fd51,BUY_POL,15m,8,0,21.421906,0.012070,0.875000,3.000000,0.897500,winner,NET_BUYER_SWING,2.913333
...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,0xf8760f160630034420b22cfb289bbb226e483812,BUY_POL,15m,4,0,3.559414,0.003069,0.500000,3.000000,0.630000,winner,NET_BUYER_SWING,5.051389
84,0xde0000094e9248c9c36f39fd2e309d40dd5c0d8c,BUY_POL,15m,4,0,2.613171,0.008350,0.500000,3.000000,0.630000,winner,NET_SELLER_SCALPER,0.527500
85,0x25398383de197587a8920b61633d331268c379b4,BUY_POL,15m,4,0,0.085562,0.004915,0.750000,1.536293,0.608629,winner,NET_SELLER_SWING,2.938056
86,0xf78ca0e0be3697c74121e54fb8163ed7d33c8e9d,BUY_POL,15m,3,0,0.245220,0.022977,0.666667,2.116220,0.604955,winner,NET_BUYER_UNKNOWN_HOLD,NaN


In [20]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_buy_1h = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["BUY_POL"],
    horizontes=["1h"],
)

display(solo_buy_1h)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0x19b5ab374071703813e8f67f2762a93e2134796c,BUY_POL,1h,17,0,30.079476,0.062095,0.882353,3.000000,0.941176,winner,NET_BUYER_UNKNOWN_HOLD,NaN
1,0x09f2f9b23e1006546ef45e5993ebef224cb4d4cd,BUY_POL,1h,13,0,8.318063,0.027794,0.846154,3.000000,0.923077,winner,NET_BUYER_SWING,1.620833
2,0xde111f1d9d78cf383d956cdfa1889e1431843b31,BUY_POL,1h,5,0,5.145673,0.043545,1.000000,3.000000,0.900000,winner,NET_BUYER_SWING,9.251111
3,0xcf9938e6408eb22c564f2c95f30581c0675ed3f7,BUY_POL,1h,5,0,1.168592,0.009727,1.000000,3.000000,0.900000,winner,NET_BUYER_UNKNOWN_HOLD,NaN
4,0x05291cfeb01f027044a88a3ced310ef903aeca10,BUY_POL,1h,23,0,6.357776,0.025036,0.782609,3.000000,0.891304,winner,NET_SELLER_SWING,1.110417
...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,0xde00000c43e2906e77dc1c60814dc91d74492dab,BUY_POL,1h,4,0,1.319090,0.017291,0.750000,1.673294,0.622329,winner,NET_BUYER_UNKNOWN_HOLD,NaN
94,0x44427e179d9aadbb8e667e3f58989e18c7a021f0,BUY_POL,1h,112,0,10.968330,0.009360,0.616071,1.117490,0.619785,winner,NET_BUYER_SWING,2.557083
95,0x4fa80a4a98c8ad6a3df0dd83a1a9e64f7297d1f9,BUY_POL,1h,59,0,28.275796,0.007553,0.593220,1.222181,0.618828,winner,NET_SELLER_SWING,1.830417
96,0x9c2f8bb814ecc6c1ece80e5bce31b0d4abb4b098,BUY_POL,1h,22,0,0.920018,0.010073,0.545455,1.356607,0.608388,winner,NET_SELLER_SCALPER,0.469861


## ACCION POR VENTA 

In [28]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_sell_1m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["SELL_POL"],
    horizontes=["1m"],
)

display(solo_sell_1m)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0x5f4674311dcd7b3e29e9cfa396d20d21b7135f2b,SELL_POL,1m,4,0,4.291951,0.001214,1.000000,3.000000,0.880000,winner,NET_BUYER_SCALPER,0.949028
1,0x44c2a2a5e0536ded1bdd8cb99f0dc5f8386de7fc,SELL_POL,1m,7,0,10.022189,0.001137,0.857143,3.000000,0.868571,winner,NET_SELLER_SWING,1.615000
2,0xaaa11e21da29cc49f856b343447d971611383048,SELL_POL,1m,3,0,4.215438,0.008675,1.000000,3.000000,0.860000,winner,NET_SELLER_UNKNOWN_HOLD,NaN
3,0x81d02327f767c6f7b5ec451e109950b2649fbccf,SELL_POL,1m,3,0,3.473932,0.001280,1.000000,3.000000,0.860000,winner,NET_SELLER_SWING,5.972778
4,0xaaa11a88e864218857e1c5ab77cccc84e90b506c,SELL_POL,1m,3,0,2.090256,0.003289,1.000000,3.000000,0.860000,winner,NET_SELLER_UNKNOWN_HOLD,NaN
5,0xccd704ecf2c26efe78f1788c6835306a8d11c57e,SELL_POL,1m,3,0,0.472316,0.002486,1.000000,3.000000,0.860000,winner,NET_SELLER_UNKNOWN_HOLD,NaN
6,0x88cbfecc3fe4378f711b024e9a5774212795c39a,SELL_POL,1m,17,0,11.013375,0.002689,0.705882,3.000000,0.852941,winner,NET_SELLER_SWING,1.648889
7,0x8828fc616dde170adfe2201860df81cfbe7d40d5,SELL_POL,1m,13,0,13.929755,0.002741,0.692308,3.000000,0.846154,winner,NET_SELLER_SWING,1.421806
8,0x6367b6bbc0197528966c7d93046a6517b939465c,SELL_POL,1m,8,0,1.307282,0.000779,0.750000,2.801613,0.815161,winner,NET_SELLER_SWING,6.475278
9,0x29417d1823b63a504ce8ef05e194885d07e4fd1f,SELL_POL,1m,5,0,0.244032,0.000747,0.800000,3.000000,0.800000,winner,NET_BUYER_SWING,5.332778


In [29]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_sell_5m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["SELL_POL"],
    horizontes=["5m"],
)

display(solo_sell_5m)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0x5f4674311dcd7b3e29e9cfa396d20d21b7135f2b,SELL_POL,5m,4,0,12.179422,0.016284,1.000000,3.000000,0.880000,winner,NET_BUYER_SCALPER,0.949028
1,0x6367b6bbc0197528966c7d93046a6517b939465c,SELL_POL,5m,8,0,12.975379,0.004406,0.750000,3.000000,0.835000,winner,NET_SELLER_SWING,6.475278
2,0x9a646698f3e92fff28999680377077b48b7448cd,SELL_POL,5m,30,0,40.297747,0.004889,0.633333,3.000000,0.816667,winner,NET_SELLER_SWING,1.761667
3,0x8828fc616dde170adfe2201860df81cfbe7d40d5,SELL_POL,5m,13,0,22.051400,0.002404,0.615385,3.000000,0.807692,winner,NET_SELLER_SWING,1.421806
4,0xd0788ddde14403679ce2a8040e783d1d9f47a24f,SELL_POL,5m,4,0,19.199208,0.012564,0.750000,3.000000,0.755000,winner,NET_SELLER_SWING,3.299583
5,0x9e2db50fcb209af568f680e0cbc66413d7290c9e,SELL_POL,5m,4,0,3.428379,0.001982,0.750000,3.000000,0.755000,winner,NET_SELLER_SCALPER,0.389167
6,0x21a59bd8dadab32b06d99b725cf4446699f0dde4,SELL_POL,5m,4,0,3.226022,0.006286,0.750000,3.000000,0.755000,winner,NET_SELLER_SWING,4.620417
7,0x2b6de4464d4caec0b0ad220de63e85e22f1d9a79,SELL_POL,5m,6,0,13.158552,0.004379,0.666667,3.000000,0.753333,winner,NET_BUYER_SWING,4.415833
8,0x7caa3572c1234d58df5f1972c9ec8efcf8ef6f5e,SELL_POL,5m,5,0,4.283223,0.002497,0.600000,3.000000,0.700000,winner,NET_SELLER_SWING,2.784167
9,0x356fd06d2aac9898b431ce2d98b563e3ae5217a6,SELL_POL,5m,5,0,3.676862,0.002156,0.600000,3.000000,0.700000,winner,NET_SELLER_SWING,3.681944


In [30]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_sell_15m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["SELL_POL"],
    horizontes=["15m"],
)

display(solo_sell_15m)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0xde11175012a274e8cbf6e3db7246d5a02b63f0a7,SELL_POL,15m,3,0,0.894218,0.002920,1.000000,3.000000,0.860000,winner,NET_SELLER_UNKNOWN_HOLD,NaN
1,0x2b6de4464d4caec0b0ad220de63e85e22f1d9a79,SELL_POL,15m,6,0,16.648310,0.006804,0.833333,3.000000,0.836667,winner,NET_BUYER_SWING,4.415833
2,0x8828fc616dde170adfe2201860df81cfbe7d40d5,SELL_POL,15m,13,0,25.596342,0.005475,0.692308,2.690764,0.815230,winner,NET_SELLER_SWING,1.421806
3,0x16023b717009fa1e425355f1a0fa0aaa1ca2abab,SELL_POL,15m,13,0,1.715847,0.004424,0.769231,2.131449,0.797760,winner,NET_BUYER_SCALPER,0.659861
4,0xda214972a1551a3058e44ffdabe408b507fc78d8,SELL_POL,15m,7,0,7.221016,0.000403,0.714286,3.000000,0.797143,winner,NET_SELLER_SWING,3.485694
5,0x5f4674311dcd7b3e29e9cfa396d20d21b7135f2b,SELL_POL,15m,4,0,22.211194,0.025638,0.750000,3.000000,0.755000,winner,NET_BUYER_SCALPER,0.949028
6,0x437dc6c1f537984622a3f757c85bc80ca6dc4c57,SELL_POL,15m,4,0,3.165212,0.008850,0.750000,3.000000,0.755000,winner,NET_SELLER_UNKNOWN_HOLD,NaN
7,0xe898563caf3b8ed945bd23e9f1a5eca1f395180e,SELL_POL,15m,3,0,15.108435,0.001631,0.666667,3.000000,0.693333,winner,NET_SELLER_UNKNOWN_HOLD,NaN
8,0xde111e36d4c98f97b29b57e03126dc855345fe96,SELL_POL,15m,3,0,2.680852,0.010905,0.666667,3.000000,0.693333,winner,NET_SELLER_SWING,8.588333
9,0x58c53d90658b8319bad56e30e2492615254257cb,SELL_POL,15m,3,0,0.584929,0.001662,0.666667,3.000000,0.693333,winner,NET_SELLER_SWING,1.883333


In [31]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_sell_1h = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["SELL_POL"],
    horizontes=["1h"],
)

display(solo_sell_1h)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0x411848a8ecc04f5ba267f57462ebee8b376a53a9,SELL_POL,1h,3,0,17.063363,0.040131,1.000000,3.000000,0.860000,winner,NET_BUYER_SWING,3.187639
1,0xde000009070a01bea4dff5dbbbe8506677ec933f,SELL_POL,1h,3,0,5.657714,0.011025,1.000000,3.000000,0.860000,winner,NET_SELLER_SCALPER,0.281806
2,0xe8696676836ec709ac693eab66c4cfc287b64496,SELL_POL,1h,14,0,63.197254,0.009948,0.642857,3.000000,0.821429,winner,NET_SELLER_SWING,1.980833
3,0xda214972a1551a3058e44ffdabe408b507fc78d8,SELL_POL,1h,7,0,14.139162,0.018572,0.714286,3.000000,0.797143,winner,NET_SELLER_SWING,3.485694
4,0x99307be0d931075780e92bfd2910374767aac01f,SELL_POL,1h,4,0,35.686305,0.011823,0.750000,3.000000,0.755000,winner,NET_SELLER_SCALPER,0.085556
5,0x2accae8f6fe7f389f03b6c18ed1d435829e32cc5,SELL_POL,1h,4,0,30.158447,0.035216,0.750000,3.000000,0.755000,winner,NET_SELLER_SWING,6.596667
6,0xae11cc8779da8f698a54653789acfc3d3e1efc5b,SELL_POL,1h,4,0,12.574860,0.017753,0.750000,3.000000,0.755000,winner,NET_SELLER_SCALPER,0.876667
7,0x21a59bd8dadab32b06d99b725cf4446699f0dde4,SELL_POL,1h,4,0,7.152117,0.023487,0.750000,3.000000,0.755000,winner,NET_SELLER_SWING,4.620417
8,0xc4fc440096b6ae0ea229ad90b6682ce795e57cd6,SELL_POL,1h,4,0,1.314997,0.003093,0.750000,3.000000,0.755000,winner,NET_SELLER_UNKNOWN_HOLD,NaN
9,0x8e0176f0332461b5cc5aa0cd2627ba0f4ca70fa8,SELL_POL,1h,7,0,9.075335,0.001048,0.571429,2.999244,0.725639,winner,NET_SELLER_UNKNOWN_HOLD,NaN


## ACCION HOLD 

In [25]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_hold_1m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["HOLD"],
    horizontes=["1m"],
)

display(solo_hold_1m)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0x890e77985a42d9bb282e195a688005459df8fd51,HOLD,1m,11,0,7.039482,0.003216,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,2.913333
1,0x064ad6bf448424ebc78e5f315b9cc8001ddb5f7c,HOLD,1m,27,0,1.962434,0.006142,0.925926,3.000000,0.962963,winner,NET_SELLER_SWING,1.311944
2,0x16023b717009fa1e425355f1a0fa0aaa1ca2abab,HOLD,1m,18,0,1.683365,0.003469,0.833333,3.000000,0.916667,winner,NET_BUYER_SCALPER,0.659861
3,0xe8696676836ec709ac693eab66c4cfc287b64496,HOLD,1m,21,0,11.004688,0.006654,0.809524,3.000000,0.904762,winner,NET_SELLER_SWING,1.980833
4,0x139618a4991d7dda7a8422e6807bd82d5f681664,HOLD,1m,5,0,0.052340,0.003637,1.000000,3.000000,0.900000,winner,NET_BUYER_SWING,3.478333
5,0x1daa4f9e8f68b71541ed4eb08f496c8e591a6bb5,HOLD,1m,4,0,0.548415,0.003172,1.000000,3.000000,0.880000,winner,NET_SELLER_SCALPER,0.513056
6,0xda214972a1551a3058e44ffdabe408b507fc78d8,HOLD,1m,4,0,0.447549,0.002445,1.000000,3.000000,0.880000,winner,NET_SELLER_SWING,3.485694
7,0xae98f425b4fb3b208160ec31903b3def960e1602,HOLD,1m,4,0,0.395950,0.000607,1.000000,3.000000,0.880000,winner,NET_SELLER_SWING,2.975556
8,0xc5a6b99696a4ef5fe76807376533799fbfddb3b1,HOLD,1m,4,0,0.039893,0.004313,1.000000,3.000000,0.880000,winner,NET_SELLER_SWING,2.435556
9,0x29417d1823b63a504ce8ef05e194885d07e4fd1f,HOLD,1m,7,0,0.431085,0.002694,0.857143,3.000000,0.868571,winner,NET_BUYER_SWING,5.332778


In [27]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_hold_5m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["HOLD"],
    horizontes=["5m"],
)

display(solo_hold_5m)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0xe8696676836ec709ac693eab66c4cfc287b64496,HOLD,5m,21,0,28.965624,0.014772,1.000000,3.000000,1.000000,winner,NET_SELLER_SWING,1.980833
1,0x890e77985a42d9bb282e195a688005459df8fd51,HOLD,5m,11,0,13.093970,0.011043,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,2.913333
2,0xd0d08887e8a5b16049534a7f3fc1de92848f5bea,HOLD,5m,75,0,12.690825,0.005168,0.986667,3.000000,0.993333,winner,NET_BUYER_SWING,8.319444
3,0x34ba113f7369e5d6d516d7e32f1206bba989511e,HOLD,5m,8,0,6.642095,0.020303,1.000000,3.000000,0.960000,winner,NET_BUYER_SWING,1.641806
4,0x2b6de4464d4caec0b0ad220de63e85e22f1d9a79,HOLD,5m,11,0,12.862675,0.013671,0.909091,3.000000,0.954545,winner,NET_BUYER_SWING,4.415833
5,0x09f2f9b23e1006546ef45e5993ebef224cb4d4cd,HOLD,5m,10,0,0.852657,0.003144,0.900000,3.000000,0.950000,winner,NET_BUYER_SWING,1.620833
6,0xe17cd5ee4152e195d4ed389ad575d09f3c3e980d,HOLD,5m,7,0,3.747558,0.009501,1.000000,3.000000,0.940000,winner,NET_BUYER_SWING,5.460278
7,0x356219ffea031c7345ce5bb1f8b29151be1f2c63,HOLD,5m,6,0,0.903568,0.011373,1.000000,3.000000,0.920000,winner,NET_SELLER_SWING,1.327500
8,0x1e3eee5b4b2811fa4798373c100d8cb16f7b957b,HOLD,5m,6,0,0.418441,0.019178,1.000000,3.000000,0.920000,winner,NET_BUYER_SWING,2.781944
9,0x4ae8f9294dcf802d22af1ccb1e05e8fc7ac26bb2,HOLD,5m,6,0,0.183744,0.002659,1.000000,3.000000,0.920000,winner,NET_BUYER_SWING,2.222083


In [32]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_hold_15m = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["HOLD"],
    horizontes=["15m"],
)

display(solo_hold_15m)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0xd0d08887e8a5b16049534a7f3fc1de92848f5bea,HOLD,15m,75,0,30.525063,0.013723,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,8.319444
1,0x2b6de4464d4caec0b0ad220de63e85e22f1d9a79,HOLD,15m,11,0,24.401030,0.023399,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,4.415833
2,0x890e77985a42d9bb282e195a688005459df8fd51,HOLD,15m,11,0,19.028890,0.014084,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,2.913333
3,0x9ecf396b3c321d4c6aa84d9720789320cc83468b,HOLD,15m,190,0,229.824617,0.019178,0.978947,3.000000,0.989474,winner,NET_BUYER_SWING,2.314444
4,0xe8296e12c13ab04c822e3693d033fb3fce77bf5d,HOLD,15m,9,0,2.433281,0.025209,1.000000,3.000000,0.980000,winner,NET_SELLER_SWING,3.326944
...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,0x25398383de197587a8920b61633d331268c379b4,HOLD,15m,5,0,0.189470,0.005116,0.600000,2.485899,0.648590,winner,NET_SELLER_SWING,2.938056
62,0xdcbc90de0e271ae34f95b53411ab4ea8c1b40401,HOLD,15m,222,0,10.460602,0.000805,0.527027,1.774660,0.640979,winner,NET_SELLER_SCALPER,0.470417
63,0x44c2a2a5e0536ded1bdd8cb99f0dc5f8386de7fc,HOLD,15m,4,0,4.317434,0.005067,0.500000,3.000000,0.630000,winner,NET_SELLER_SWING,1.615000
64,0x411848a8ecc04f5ba267f57462ebee8b376a53a9,HOLD,15m,4,0,0.410670,0.002031,0.500000,3.000000,0.630000,winner,NET_BUYER_SWING,3.187639


In [33]:
from defi4.wallets import filtrar_wallets_ganadoras

solo_hold_1h = filtrar_wallets_ganadoras(
    resultado.perfiles,
    acciones=["HOLD"],
    horizontes=["1h"],
)

display(solo_hold_1h)

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
0,0xd0d08887e8a5b16049534a7f3fc1de92848f5bea,HOLD,1h,75,0,86.655102,0.041896,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,8.319444
1,0x064ad6bf448424ebc78e5f315b9cc8001ddb5f7c,HOLD,1h,17,0,7.817119,0.009126,1.000000,3.000000,1.000000,winner,NET_SELLER_SWING,1.311944
2,0x26bd98954782c5f8ae5cb6e6f6c8617abacfd80a,HOLD,1h,10,0,2.165114,0.044677,1.000000,3.000000,1.000000,winner,NET_SELLER_SCALPER,0.729583
3,0x9ecf396b3c321d4c6aa84d9720789320cc83468b,HOLD,1h,165,0,358.738942,0.030699,0.987879,3.000000,0.993939,winner,NET_BUYER_SWING,2.314444
4,0x574be013006558d0a5682a6b0508f02d147377bf,HOLD,1h,15,0,20.760452,0.072802,0.933333,3.000000,0.966667,winner,NET_SELLER_SCALPER,0.838611
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,0xae98f425b4fb3b208160ec31903b3def960e1602,HOLD,1h,4,0,5.914182,0.017631,0.500000,3.000000,0.630000,winner,NET_SELLER_SWING,2.975556
60,0x2a818a31d9e4cc51217adda95b2b754acd9e58f1,HOLD,1h,111,0,2.324970,0.002350,0.513514,1.654651,0.622222,winner,NET_BUYER_SWING,3.220833
61,0xb4b919ddea467cf55fbf0d2c28cf3eafc9b1da00,HOLD,1h,661,0,8.230967,0.005476,0.534039,1.535881,0.620608,winner,NET_SELLER_SCALPER,0.817500
62,0x025a9c987abc1d04f15335e0d2f8dde3e246a7ae,HOLD,1h,84,0,2.076134,0.016773,0.607143,1.066758,0.610247,winner,NET_BUYER_SWING,1.440972


In [35]:
from defi4.pipeline import ejecutar_desde_parquet

resultado = ejecutar_desde_parquet(
    parquet_path="cache/swaps_24h.parquet",
    output_dir="informes_pol",
    min_decisiones=3,
    export_png=False,
)



Informe creado sin red: informes_pol\snapshot_20260823T014719Z_08


In [41]:
solo_winners= resultado.perfiles[resultado.perfiles["winner_status"]=="winner"]
solo_winners


,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas
12847,0x064ad6bf448424ebc78e5f315b9cc8001ddb5f7c,HOLD,1h,17,0,7.817119,0.009126,1.000000,3.000000,1.000000,winner,NET_SELLER_SWING,1.311944
12848,0x26bd98954782c5f8ae5cb6e6f6c8617abacfd80a,HOLD,1h,10,0,2.165114,0.044677,1.000000,3.000000,1.000000,winner,NET_SELLER_SCALPER,0.729583
12849,0x2b6de4464d4caec0b0ad220de63e85e22f1d9a79,HOLD,15m,11,0,24.401030,0.023399,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,4.415833
12850,0x890e77985a42d9bb282e195a688005459df8fd51,HOLD,15m,11,0,19.028890,0.014084,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,2.913333
12851,0x890e77985a42d9bb282e195a688005459df8fd51,HOLD,1m,11,0,7.039482,0.003216,1.000000,3.000000,1.000000,winner,NET_BUYER_SWING,2.913333
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13473,0xb2bffc3788acfb35bf6555fa1fe6adc2c0afe9e5,BUY_POL,1m,120,0,1.857759,0.000453,0.575000,1.178249,0.605325,winner,NET_SELLER_SCALPER,0.717222
13474,0xf78ca0e0be3697c74121e54fb8163ed7d33c8e9d,BUY_POL,15m,3,0,0.245220,0.022977,0.666667,2.116220,0.604955,winner,NET_BUYER_UNKNOWN_HOLD,NaN
13475,0x3fc0910bebc34db4c985474c4dd2f84170aee276,SELL_POL,1h,3,0,2.210758,0.013549,0.666667,2.099946,0.603328,winner,NET_BUYER_SWING,1.042639
13476,0xb886a6f9725b5c387f221b13d0efda1e66999aef,SELL_POL,5m,19,0,7.370357,0.000492,0.526316,1.401371,0.603295,winner,NET_BUYER_SWING,5.603889


# Parte III. Modelo de aprendizaje por refuerzo guiado por wallets

Esta sección transforma los perfiles on-chain ya calculados en un modelo de decisión. El objetivo no es copiar una wallet, sino aprender una política que use su evidencia histórica como una señal adicional para operar POL cada hora.

## 1. Definición teórica: proceso de decisión de Markov

Un problema de aprendizaje por refuerzo se representa como un **Proceso de Decisión de Markov (MDP)**:

$$M=(S,A,P,R,\gamma)$$

- $S$: conjunto de estados observables del mercado y del portafolio.
- $A$: acciones que puede tomar el agente.
- $P(s'|s,a)$: probabilidad de pasar al estado siguiente $s'$ después de elegir $a$ en $s$.
- $R(s,a,s')$: recompensa obtenida por la decisión.
- $\gamma$: factor de descuento; valora las recompensas futuras sin ignorar la recompensa inmediata.

Aquí el **agente** es una estrategia automática de POL; el **entorno** es el mercado POL/USDC observado on-chain; una decisión ocurre cada **1 hora**; y el episodio tiene inicialmente 24 decisiones.

## 2. Wallets ganadoras como señal, no como regla fija

Primero se construyen perfiles por `wallet × acción × horizonte`. Para el agente sólo se usan perfiles de **1 h** que ya son ganadores y tienen un umbral alto:

$$winner\_status=winner \quad \text{y} \quad consistency\_score \ge 0.80$$

El score resume tres evidencias: tasa de acierto, calidad de ganancias frente a pérdidas y número de decisiones maduras:

$$consistency = 0.50\cdot tasa\_acierto + 0.30\cdot\frac{min(profit\_factor,3)}{3} + 0.20\cdot min(n/10,1)$$

Una decisión está madura si ya llegó su precio posterior de una hora. En el corte $t$, el sistema sólo usa decisiones con precio forward conocido antes o en $t$; por eso la señal no usa información futura. Se suman los scores de las wallets Buy, Sell y Hold. La dirección con soporte único más alto genera `BUY`, `SELL` o `HOLD`; un empate o ausencia de evidencia genera `NEUTRAL`.

## 3. Estado y acciones del agente

El estado combina tres elementos:

$$s_t=(r_t,w_t,x_t)$$

- $r_t$: régimen de mercado `DOWN`, `FLAT` o `UP`. Se construye a partir de retornos pasados de 1m, 5m, 15m y 1h. Una banda fija de ±0.10 % define `FLAT`, para no depender de precios futuros.
- $w_t$: señal causal de wallets: `BUY`, `SELL`, `HOLD` o `NEUTRAL`.
- $x_t$: posición actual: `0 = USDC` y `1 = POL`.

Por tanto, hay $3\times4\times2=24$ estados posibles. El modelo es **long-only**: no vende POL que no posee ni abre posiciones cortas.

| Posición actual | Acciones admisibles | Posición siguiente |
|---|---|---|
| USDC (`0`) | `BUY_POL`, `HOLD` | POL o USDC |
| POL (`1`) | `SELL_POL`, `HOLD` | USDC o POL |

## 4. Transiciones empíricas

No se inventan probabilidades. Se observan los cambios horarios consecutivos de $(r_t,w_t)$ en los datos y se cuentan. Después se aplica suavizado de Laplace para que una transición no observada no tenga probabilidad exactamente cero:

$$P(s'|s,a)=\frac{conteo(s\rightarrow s')+\alpha}{\sum_j conteo(s\rightarrow s_j)+\alpha\cdot|S_{económico}|}$$

con $\alpha=1$. La acción determina la posición siguiente: por ejemplo, desde USDC, `BUY_POL` lleva a posición POL; desde POL, `SELL_POL` lleva a USDC. En cada estado y acción admisible verificamos que $\sum_{s'}P(s'|s,a)=1$.

## 5. Función de recompensa

La recompensa principal es siempre financiera. Si $p_t$ es el precio actual y $p_{t+1}$ el precio conocido una hora después, el retorno de POL es:

$$retorno_{POL}=\frac{p_{t+1}}{p_t}-1$$

La recompensa base es el retorno de la posición que queda después de la acción, menos gas al comprar o vender. Así, desde USDC: `BUY_POL` obtiene $retorno_{POL}-gas$ y `HOLD` obtiene 0. Desde POL: `SELL_POL` obtiene $-gas$ y `HOLD` obtiene $retorno_{POL}$. El precio $p_{t+1}$ sólo se utiliza sobre el historial para calcular recompensas y estimar el modelo; en una decisión real el agente sólo observa la información disponible en $t$.

Las wallets no entregan dinero ficticio sólo por seguirlas. Añaden un bonus pequeño únicamente si su dirección dominante coincide con una decisión que efectivamente fue mejor que su alternativa:

$$R_{final}=R_{base}+0.25\cdot confianza_{wallets}\cdot ventaja_{realizada}$$

La confianza es mayor cuando el soporte de una dirección supera claramente a las otras. La ventaja compara la recompensa base de una acción contra la otra acción admisible.

## 6. Política óptima mediante Bellman

Como el episodio tiene horizonte finito de 24 horas, se resuelve Bellman hacia atrás, desde la última hora hasta la primera:

$$V_t(s)=\max_{a\in A(s)}\left[R(s,a)+\gamma\sum_{s'}P(s'|s,a)V_{t+1}(s')\right]$$

La acción que alcanza el máximo define la política $\pi_t^*(s)$. En palabras sencillas: para cada estado, el agente compara las acciones permitidas, combina la ganancia inmediata con el valor esperado de las siguientes horas y elige la mejor. Usamos $\gamma=0.99$, por lo que el futuro importa casi tanto como la hora actual.

## 7. Ejemplo numérico sencillo

Supón que el agente está en USDC, POL vale $1.00$ ahora y $1.02$ una hora después. El retorno de POL es 2 %. Comprar cuesta 0.10 % de gas. Las wallets confirmadas suman soporte Buy = 0.90, Sell = 0.10 y Hold = 0.00.

La confianza de Buy es $(0.90-0.10)/(0.90+0.10)=0.80$. Las dos acciones permitidas son:

| Acción | Recompensa base | Ventaja frente a la alternativa | Bonus wallets | Recompensa final |
|---|---:|---:|---:|---:|
| `BUY_POL` | $0.02-0.001=0.019$ | $0.019-0=0.019$ | $0.25\times0.80\times0.019=0.0038$ | $0.0228$ |
| `HOLD` en USDC | $0$ | $0-0.019=-0.019$ | $0$ | $0$ |

En esta hora Buy es mejor porque el precio subió y las wallets Buy tenían apoyo dominante. Si el precio hubiera caído, la recompensa base de comprar sería negativa y el bonus tampoco convertiría esa mala decisión en una buena.

## 8. Interpretación y límites

Las siguientes celdas ejecutan el modelo con los datos disponibles y muestran la cohorte de wallets, los estados, la verificación de transiciones, la política y un replay histórico. El replay es una explicación **in-sample**, no una prueba de rentabilidad real. La ventana de 24 horas es aún pequeña para estimar una política estable; el siguiente paso es acumular snapshots, separar entrenamiento/validación/prueba en orden temporal y evaluar fuera de muestra. El modelo tampoco representa balances externos, posiciones cortas ni impacto de mercado.

In [47]:
winner_filtrados = solo_winners[solo_winners["consistency_score"]>=0.8]
winner_filtrados["horizonte"].value_counts()

15m    84
1h     82
5m     71
1m     42
Name: horizonte, dtype: int64

In [52]:
winner_filtrados[["horizonte","retorno_neto_mediano"]].groupby(by="horizonte").mean()


,retorno_neto_mediano
horizonte,
15m,0.015129
1h,0.027162
1m,0.003391
5m,0.007429


In [53]:
# 1. Wallets ganadoras dirigidas: sólo 1h y consistency_score >= 0.80
from defi4.wallets import calcular_wallets_ganadoras_1h_consistentes

UMBRAL_CONSISTENCIA = 0.80

wallets_dirigidas_1h = calcular_wallets_ganadoras_1h_consistentes(
    resultado.perfiles,
    consistency_threshold=UMBRAL_CONSISTENCIA,
)

display(wallets_dirigidas_1h)
print("Wallets únicas:", wallets_dirigidas_1h["wallet"].nunique())

,wallet,accion,horizonte,n_decisiones,n_pendientes,pnl_neto_usdc,retorno_neto_mediano,tasa_acierto,profit_factor,consistency_score,winner_status,perfil_conductual,duration_hold_mediana_horas,direccion_agente
0,0xd0d08887e8a5b16049534a7f3fc1de92848f5bea,HOLD,1h,75,0,86.655102,0.041896,1.000000,3.0,1.000000,winner,NET_BUYER_SWING,8.319444,HOLD
1,0x064ad6bf448424ebc78e5f315b9cc8001ddb5f7c,HOLD,1h,17,0,7.817119,0.009126,1.000000,3.0,1.000000,winner,NET_SELLER_SWING,1.311944,HOLD
2,0x26bd98954782c5f8ae5cb6e6f6c8617abacfd80a,HOLD,1h,10,0,2.165114,0.044677,1.000000,3.0,1.000000,winner,NET_SELLER_SCALPER,0.729583,HOLD
3,0x9ecf396b3c321d4c6aa84d9720789320cc83468b,HOLD,1h,165,0,358.738942,0.030699,0.987879,3.0,0.993939,winner,NET_BUYER_SWING,2.314444,HOLD
4,0x574be013006558d0a5682a6b0508f02d147377bf,HOLD,1h,15,0,20.760452,0.072802,0.933333,3.0,0.966667,winner,NET_SELLER_SCALPER,0.838611,HOLD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,0xe8696676836ec709ac693eab66c4cfc287b64496,SELL_POL,1h,14,0,63.197254,0.009948,0.642857,3.0,0.821429,winner,NET_SELLER_SWING,1.980833,SELL
78,0xb886a6f9725b5c387f221b13d0efda1e66999aef,HOLD,1h,30,0,27.950317,0.012497,0.633333,3.0,0.816667,winner,NET_BUYER_SWING,5.603889,HOLD
79,0x324941adbbba8f3ef23962626e5e05a75d4a7094,BUY_POL,1h,130,0,15.542896,0.012495,0.630769,3.0,0.815385,winner,NET_SELLER_SCALPER,0.275694,BUY
80,0x26bd98954782c5f8ae5cb6e6f6c8617abacfd80a,BUY_POL,1h,35,0,6.474732,0.004228,0.600000,3.0,0.800000,winner,NET_SELLER_SCALPER,0.729583,BUY


Wallets únicas: 66


In [54]:
# 2. Construir el MDP y resolver Bellman sin consultar Alchemy otra vez
from defi4.model import ejecutar_rl_desde_snapshot

resultado_rl = ejecutar_rl_desde_snapshot(
    snapshot_dir=resultado.snapshot_dir,
    output_dir="informes_rl",
    consistency_threshold=0.80,
    flat_band=0.001,
    profile_weight=0.25,
    horizon=24,
    gamma=0.99,
)

print("Artefactos RL:", resultado_rl.output_dir)

Artefactos RL: informes_rl\snapshot_20260823T010000Z_rl_1h_02


In [55]:
# 3. Estados horarios: mercado + señal de wallets + posición
observaciones_rl = resultado_rl.observaciones

display(
    observaciones_rl[
        [
            "as_of", "precio_t", "precio_t1",
            "regimen_mercado", "senal_wallets",
            "confianza_wallets"
        ]
    ]
)

,as_of,precio_t,precio_t1,regimen_mercado,senal_wallets,confianza_wallets
0,2026-08-22 02:00:00+00:00,0.095400,0.098968,FLAT,NEUTRAL,0.000000
1,2026-08-22 03:00:00+00:00,0.098968,0.104991,UP,BUY,1.000000
2,2026-08-22 04:00:00+00:00,0.104991,0.110602,UP,HOLD,0.133903
3,2026-08-22 05:00:00+00:00,0.110602,0.107340,UP,HOLD,0.074561
4,2026-08-22 06:00:00+00:00,0.107340,0.112344,DOWN,HOLD,0.188495
5,2026-08-22 07:00:00+00:00,0.112344,0.113131,UP,HOLD,0.104245
6,2026-08-22 08:00:00+00:00,0.113131,0.118690,UP,HOLD,0.069155
7,2026-08-22 09:00:00+00:00,0.118690,0.113208,UP,BUY,0.003858
8,2026-08-22 10:00:00+00:00,0.113208,0.110231,DOWN,BUY,0.007623
9,2026-08-22 11:00:00+00:00,0.110231,0.111842,DOWN,HOLD,0.242417


In [56]:
# 4. Verificar que P(s' | s, a) suma 1
from defi4.model import verificar_probabilidades

verificacion = verificar_probabilidades(resultado_rl.mdp)
display(verificacion)

assert verificacion["valida"].all()
print("Todas las transiciones válidas suman 1.")

,estado,accion,suma_probabilidades,valida
0,DOWN | BUY | posición=0,BUY_POL,1.0,True
1,DOWN | BUY | posición=0,HOLD,1.0,True
2,DOWN | BUY | posición=1,SELL_POL,1.0,True
3,DOWN | BUY | posición=1,HOLD,1.0,True
4,DOWN | SELL | posición=0,BUY_POL,1.0,True
5,DOWN | SELL | posición=0,HOLD,1.0,True
6,DOWN | SELL | posición=1,SELL_POL,1.0,True
7,DOWN | SELL | posición=1,HOLD,1.0,True
8,DOWN | HOLD | posición=0,BUY_POL,1.0,True
9,DOWN | HOLD | posición=0,HOLD,1.0,True


Todas las transiciones válidas suman 1.


In [57]:
# 5. Política óptima calculada con Bellman
politica = resultado_rl.politica

display(
    politica.sort_values(
        ["posicion", "regimen_mercado", "senal_wallets"]
    )
)

,regimen_mercado,senal_wallets,posicion,accion_recomendada,valor_optimo
0,DOWN,BUY,0,HOLD,0.161042
4,DOWN,HOLD,0,BUY_POL,0.163058
6,DOWN,NEUTRAL,0,BUY_POL,0.166004
2,DOWN,SELL,0,BUY_POL,0.166004
8,FLAT,BUY,0,BUY_POL,0.166004
12,FLAT,HOLD,0,BUY_POL,0.179072
14,FLAT,NEUTRAL,0,BUY_POL,0.197972
10,FLAT,SELL,0,BUY_POL,0.166004
16,UP,BUY,0,BUY_POL,0.174372
20,UP,HOLD,0,BUY_POL,0.161254


In [58]:
# 6. Replay histórico: qué habría decidido el agente cada hora
replay = resultado_rl.replay

display(replay)
print("Riqueza final, base 100:", replay["wealth"].iloc[-1])

,as_of,estado,accion,posicion_despues,recompensa_base,bonus_wallets,recompensa_final,wealth
0,2026-08-22 02:00:00+00:00,FLAT | NEUTRAL | posición=0,BUY_POL,1,0.036647,0.000000,0.036647,103.664699
1,2026-08-22 03:00:00+00:00,UP | BUY | posición=1,HOLD,1,0.060858,0.000000,0.060858,109.973480
2,2026-08-22 04:00:00+00:00,UP | HOLD | posición=1,HOLD,1,0.053438,0.001814,0.055252,116.049734
3,2026-08-22 05:00:00+00:00,UP | HOLD | posición=1,HOLD,1,-0.029488,-0.000535,-0.030023,112.565559
4,2026-08-22 06:00:00+00:00,DOWN | HOLD | posición=1,HOLD,1,0.046618,0.002233,0.048851,118.064483
5,2026-08-22 07:00:00+00:00,UP | HOLD | posición=1,HOLD,1,0.006997,0.000202,0.007199,118.914394
6,2026-08-22 08:00:00+00:00,UP | HOLD | posición=1,HOLD,1,0.049141,0.000863,0.050004,124.860606
7,2026-08-22 09:00:00+00:00,UP | BUY | posición=1,HOLD,1,-0.046186,-0.000000,-0.046186,119.093843
8,2026-08-22 10:00:00+00:00,DOWN | BUY | posición=1,SELL_POL,0,-0.000761,0.000000,-0.000761,119.003177
9,2026-08-22 11:00:00+00:00,DOWN | HOLD | posición=0,BUY_POL,1,0.013852,0.000000,0.013852,120.651567


Riqueza final, base 100: 113.96802853509203


In [62]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "iframe"

fig_senal = px.scatter(
    observaciones_rl,
    x="as_of",
    y="confianza_wallets",
    color="senal_wallets",
    hover_data=["support_buy", "support_sell", "support_hold"],
    title="Señal causal de wallets ganadoras por hora",
)

fig_replay = px.line(
    replay,
    x="as_of",
    y="wealth",
    markers=True,
    title="Replay histórico de la política Bellman",
)

fig_senal.show()
fig_replay.show()